# LeetCode 3571 — Find the Shortest Superstring II bằng Python

Link đề: https://leetcode.com/problems/find-the-shortest-superstring-ii/ · Độ khó: **Easy** · Chủ đề: String, Two Pointers

> **Bài Premium:** đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium. (Tên bài, chữ ký hàm, hai ví dụ và constraints dưới đây khớp với nguồn công khai; đề gốc cho phép trả về **bất kỳ** đáp án ngắn nhất, vì vậy bộ test của notebook kiểm tra **độ dài và việc chứa cả hai chuỗi**, không so sánh chuỗi chính xác.)

Skeleton LeetCode:

```python
class Solution:
    def shortestSuperstring(self, s1: str, s2: str) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm chuỗi **ngắn nhất** chứa **cả hai** chuỗi `s1`, `s2` như các **chuỗi con liên tiếp (substring)**; nhiều đáp án thì trả một đáp án bất kỳ.
2. Hiểu khái niệm **độ chồng lấp (overlap)**: phần **hậu tố** của chuỗi này trùng với **tiền tố** của chuỗi kia, và vì sao gộp chồng lấp lớn nhất cho chuỗi ngắn nhất.
3. Biết dùng `in`, cắt lát, `endswith`, `startswith`, và hàm tiền tố **KMP** (prefix function) để tính overlap.
4. Viết 4 cách: thử overlap từ lớn xuống nhỏ, quét điểm bắt đầu của hậu tố, **trượt `s2` theo mọi độ lệch** trên `s1`, và KMP; đối chiếu chúng bằng test ngẫu nhiên và một **oracle vét cạn** cho chuỗi nhỏ.
5. Tránh các lỗi: chỉ nối `s1 + s2`, bỏ qua trường hợp một chuỗi nằm trong chuỗi kia, chỉ thử một thứ tự, và chọn overlap **nhỏ nhất** thay vì lớn nhất.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s1` và `s2`. Trả về chuỗi **ngắn nhất** chứa **cả `s1` lẫn `s2`** dưới dạng chuỗi con (substring: dãy ký tự liên tiếp). Nếu có nhiều đáp án hợp lệ, trả về **bất kỳ** đáp án nào.

| Ví dụ | s1, s2 | Output | Giải thích |
|---|---|---|---|
| 1 | `"aba"`, `"bab"` | `"abab"` | `"abab"` chứa `"aba"` (vị trí 0..2) và `"bab"` (vị trí 1..3), dài `4` |
| 2 | `"aa"`, `"aaa"` | `"aaa"` | `"aa"` đã nằm trong `"aaa"` nên chỉ cần `"aaa"` |

Constraints: `1 <= s1.length, s2.length <= 100`; `s1`, `s2` chỉ gồm chữ cái tiếng Anh thường.

Chú ý: với ví dụ 1, `"baba"` cũng hợp lệ (chứa `"bab"` ở 0..2 và `"aba"` ở 1..3). Vì vậy khi kiểm tra ta so **độ dài** và kiểm tra **chứa cả hai**, không so từng ký tự với một đáp án mẫu.

In [1]:
for s1, s2 in [("aba", "bab"), ("aa", "aaa")]:
    for cand in ["abab", "baba", "aaa", "ababab", "abaa"]:
        ok = s1 in cand and s2 in cand
        print(f"s1={s1!r}, s2={s2!r}, ứng viên {cand!r}: chứa cả hai? {ok}, độ dài {len(cand)}")

s1='aba', s2='bab', ứng viên 'abab': chứa cả hai? True, độ dài 4
s1='aba', s2='bab', ứng viên 'baba': chứa cả hai? True, độ dài 4
s1='aba', s2='bab', ứng viên 'aaa': chứa cả hai? False, độ dài 3
s1='aba', s2='bab', ứng viên 'ababab': chứa cả hai? True, độ dài 6
s1='aba', s2='bab', ứng viên 'abaa': chứa cả hai? False, độ dài 4
s1='aa', s2='aaa', ứng viên 'abab': chứa cả hai? False, độ dài 4
s1='aa', s2='aaa', ứng viên 'baba': chứa cả hai? False, độ dài 4
s1='aa', s2='aaa', ứng viên 'aaa': chứa cả hai? True, độ dài 3
s1='aa', s2='aaa', ứng viên 'ababab': chứa cả hai? False, độ dài 6
s1='aa', s2='aaa', ứng viên 'abaa': chứa cả hai? False, độ dài 4


## 3. Kiến thức nền

### 3.1. Chuỗi con (substring) và toán tử `in`

Chuỗi con là dãy ký tự **liên tiếp**. `a in b` kiểm tra `a` có là chuỗi con của `b` không. Khác với **dãy con (subsequence)** — có thể bỏ ký tự ở giữa. Bài này chỉ dùng chuỗi con liên tiếp. Ví dụ `"ab" in "xaby"` là `True`, nhưng `"ay" in "xaby"` là `False`.

In [2]:
print("ab" in "xaby", "ay" in "xaby", "aa" in "aaa", "aaa" in "aa")

True False True False


### 3.2. Hậu tố, tiền tố và cắt lát

**Tiền tố** (prefix) là phần đầu: `s[:k]`. **Hậu tố** (suffix) là phần cuối: `s[len(s) - k:]` (cẩn thận `s[-0:]` là cả chuỗi, không phải rỗng!). `s.startswith(p)` và `s.endswith(p)` kiểm tra nhanh.

In [3]:
s = "abcde"
print(s[:2], s[len(s) - 2:], s.startswith("ab"), s.endswith("de"))
print(repr(s[len(s) - 0:]), "<- k = 0 cho chuỗi rỗng")
print(repr(s[-0:]), "<- s[-0:] là CẢ chuỗi, dễ nhầm")

ab de True True
'' <- k = 0 cho chuỗi rỗng
'abcde' <- s[-0:] là CẢ chuỗi, dễ nhầm


### 3.3. Overlap (độ chồng lấp)

`overlap(a, b)` = độ dài `k` **lớn nhất** sao cho hậu tố `k` ký tự cuối của `a` bằng tiền tố `k` ký tự đầu của `b`. Ví dụ `a = "aba"`, `b = "bab"`: hậu tố `"ba"` của `a` bằng tiền tố `"ba"` của `b` nên `overlap = 2`. Khi đó ta **gộp** thành `a + b[k:] = "aba" + "b" = "abab"` (dài `3 + 3 - 2 = 4`).

In [4]:
def overlap(a, b):
    for k in range(min(len(a), len(b)), -1, -1):      # từ lớn xuống nhỏ; k = 0 luôn đúng
        if a[len(a) - k:] == b[:k]:
            return k

print(overlap("aba", "bab"), overlap("bab", "aba"), overlap("ab", "ca"), overlap("ca", "ab"))
a, b = "aba", "bab"
k = overlap(a, b)
print(a + b[k:])

2 2 0 1
abab


### 3.4. `min` với `key=len`

`min(list, key=len)` trả về phần tử có độ dài nhỏ nhất; nếu hòa thì trả phần tử **xuất hiện trước**. Ta dùng nó để chọn giữa hai thứ tự ghép `(s1, s2)` và `(s2, s1)`.

In [5]:
print(min(["abab", "baba"], key=len), min(["abc", "xy", "pq"], key=len))

abab xy


### 3.5. Hàm tiền tố (prefix function / mảng `pi`) của KMP

Với chuỗi `t`, `pi[i]` là độ dài **tiền tố dài nhất (nhỏ hơn `i+1`)** của `t[:i+1]` đồng thời là **hậu tố** của nó. Nhờ vậy overlap của `a` (hậu tố) và `b` (tiền tố) có thể lấy từ `pi` của chuỗi ghép `b + "#" + a` (ký tự `#` không có trong chữ thường, ngăn kết quả vượt quá `len(b)`): `overlap(a, b) = pi[-1]`. Tính `pi` mất `O(len(t))`.

In [6]:
def prefix_function(t):
    pi = [0] * len(t)
    for i in range(1, len(t)):
        k = pi[i - 1]
        while k > 0 and t[i] != t[k]:
            k = pi[k - 1]
        if t[i] == t[k]:
            k += 1
        pi[i] = k
    return pi

print(prefix_function("abab"), prefix_function("bab#aba"))
print(prefix_function("bab#aba")[-1], "= overlap('aba', 'bab')")

[0, 0, 1, 2] [0, 0, 1, 0, 0, 1, 2]
2 = overlap('aba', 'bab')


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gộp `s1 = "aba"` và `s2 = "bab"`. Có hai cách "dán" chúng lại:

```
s1 trước, s2 sau:      aba            overlap(s1, s2) = 2 (hậu tố "ba" = tiền tố "ba")
                        bab           ->  a b a b     = "abab"   (dài 4)

s2 trước, s1 sau:      bab            overlap(s2, s1) = 2 (hậu tố "ab" = tiền tố "ab")
                        aba           ->  b a b a     = "baba"   (dài 4)
```

Chồng lấp càng nhiều thì chuỗi càng ngắn: độ dài = `len(s1) + len(s2) - overlap`. Hai thứ tự cho kết quả có thể khác nhau, nên cần thử cả hai rồi lấy ngắn hơn.

Có một trường hợp đặc biệt: nếu `s1` **nằm trọn trong** `s2` (hoặc ngược lại) thì chuỗi dài hơn đã chứa chuỗi kia và là đáp án. Ví dụ `"aa"` nằm trong `"aaa"`. Đây là trường hợp mà "chỉ dán hậu tố–tiền tố" không phát hiện (chuỗi nằm **giữa**), nên ta phải kiểm tra `in` trước.

## 5. Cách 1 — Brute Force / baseline

1. Nếu `s1 in s2`: trả `s2`. Nếu `s2 in s1`: trả `s1`.
2. Với mỗi thứ tự `(a, b) ∈ {(s1, s2), (s2, s1)}`: thử `k` từ `min(len(a), len(b))` **xuống** `0`, lấy `k` đầu tiên thỏa `a[len(a) - k:] == b[:k]`, rồi dán `a + b[k:]`.
3. Trả chuỗi ngắn hơn trong hai kết quả (hòa thì lấy kết quả của `(s1, s2)`).

Dán với overlap lớn nhất cho chuỗi ngắn nhất cho thứ tự đó, vì độ dài là `len(a) + len(b) - k`.

In [7]:
class SolutionOverlap:
    def shortestSuperstring(self, s1, s2):
        if s1 in s2:
            return s2
        if s2 in s1:
            return s1

        def merge(a, b):
            for k in range(min(len(a), len(b)), -1, -1):
                if a[len(a) - k:] == b[:k]:
                    return a + b[k:]

        return min(merge(s1, s2), merge(s2, s1), key=len)

print(SolutionOverlap().shortestSuperstring("aba", "bab"))
print(SolutionOverlap().shortestSuperstring("aa", "aaa"))
print(SolutionOverlap().shortestSuperstring("ab", "ca"))

abab
aaa
cab


**Độ phức tạp:** gọi `m = len(s1)`, `n = len(s2)`. `in` mất `O(m·n)` trong trường hợp xấu (hoặc nhanh hơn trong CPython); vòng lặp `k` tối đa `min(m, n)` lần, mỗi lần so sánh/cắt `O(k)` -> `O(min(m, n)²)`. Với `m, n <= 100` rất nhanh. Bộ nhớ `O(m + n)` cho chuỗi kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Mọi cách đặt hai chuỗi lên một đường thẳng** được mô tả bằng **độ lệch** `t` của `s2` so với `s1`: `s1` chiếm vị trí `0..m-1`, `s2` chiếm `t..t+n-1`. Hai chuỗi phải **khớp** ở mọi vị trí mà cả hai cùng phủ. Độ dài tổng là `max(m, t+n) - min(0, t)`. Ta muốn độ dài nhỏ nhất.

**Bước đột phá (góc nhìn):** chỉ có `m + n` độ lệch đáng xét (từ `-(n-1)` đến `m`), nên có thể **vét cạn** các độ lệch, kiểm tra khớp, lấy cái ngắn nhất. Các độ lệch hợp lệ chia làm ba loại:

- `t` sao cho `s2` nằm trọn trong `s1` -> chứa nhau (đáp án dài `m`);
- `t > 0` và `t + n > m` (s2 thò ra bên phải): chồng lấp là hậu tố của `s1`/tiền tố của `s2`;
- `t < 0` (s2 thò ra bên trái): chồng lấp là hậu tố của `s2`/tiền tố của `s1`;
- `t >= m`: không chồng lấp (nối thẳng, `t = m`) — luôn khớp.

**Quy tắc "nếu… thì…":**

- Nếu `s1 in s2` hoặc `s2 in s1` thì đáp án là chuỗi dài hơn.
- Ngược lại, độ dài ngắn nhất = `m + n - max(overlap(s1, s2), overlap(s2, s1))`.
- Overlap tính được bằng vét cạn `O(min(m,n)²)` hoặc bằng KMP `O(m + n)`.

In [8]:
s1, s2 = "aba", "bab"
m, n = len(s1), len(s2)
for t in range(-(n - 1), m + 1):
    lo, hi = min(0, t), max(m, t + n)
    ok = all(s1[p] == s2[p - t] for p in range(max(0, t), min(m, t + n)))
    print(f"độ lệch t={t:>2}: {'khớp' if ok else 'không khớp'}, độ dài gộp = {hi - lo}")

độ lệch t=-2: không khớp, độ dài gộp = 5
độ lệch t=-1: khớp, độ dài gộp = 4
độ lệch t= 0: không khớp, độ dài gộp = 3
độ lệch t= 1: khớp, độ dài gộp = 4
độ lệch t= 2: không khớp, độ dài gộp = 5
độ lệch t= 3: khớp, độ dài gộp = 6


## 7. Các cách tiếp theo

Ta có 4 cách: (1) thử overlap từ lớn xuống nhỏ; (2) quét điểm bắt đầu của hậu tố; (3) trượt `s2` theo mọi độ lệch trên `s1`; (4) KMP.

### Cách 2 — Quét điểm bắt đầu của hậu tố bằng `startswith`

Hậu tố của `a` bắt đầu ở vị trí `i`; nếu `b.startswith(a[i:])` thì overlap bằng `len(a) - i`. Quét `i` từ `0` tăng dần thì hậu tố **dài nhất** (overlap lớn nhất) được gặp trước. Dùng `startswith` thay cho so sánh cắt lát.

In [9]:
class SolutionSuffixScan:
    def shortestSuperstring(self, s1, s2):
        if s1 in s2:
            return s2
        if s2 in s1:
            return s1

        def merge(a, b):
            for i in range(len(a) + 1):                 # i = len(a) -> hậu tố rỗng
                if b.startswith(a[i:]):
                    return a + b[len(a) - i:]

        r1, r2 = merge(s1, s2), merge(s2, s1)
        return r1 if len(r1) <= len(r2) else r2

print(SolutionSuffixScan().shortestSuperstring("aba", "bab"))
print(SolutionSuffixScan().shortestSuperstring("aa", "aaa"))
print(SolutionSuffixScan().shortestSuperstring("ab", "ca"))

abab
aaa
cab


Độ phức tạp Cách 2: tối đa `len(a) + 1` lần `startswith`, mỗi lần `O(len)` -> `O(m · min(m, n))`. Có thể chậm hơn Cách 1 một chút vì quét cả các hậu tố dài hơn `n` (không thể khớp), nhưng `startswith` thoát nhanh.

### Cách 3 — Trượt `s2` theo mọi độ lệch trên `s1`

Duyệt độ lệch `t` từ `-(n-1)` đến `m`; kiểm tra khớp trên phần hai chuỗi cùng phủ (đã gồm trường hợp `s2` nằm trong `s1`, `s1` nằm trong `s2`, chồng lấp hai phía và nối thẳng). Với mỗi `t` khớp, dựng chuỗi gộp: ký tự tại vị trí `p` lấy từ `s1` nếu `p` thuộc `[0, m)`, ngược lại từ `s2`. Giữ chuỗi ngắn nhất.

In [10]:
class SolutionOffset:
    def shortestSuperstring(self, s1, s2):
        m, n = len(s1), len(s2)
        best = None
        for t in range(-(n - 1), m + 1):
            if not all(s1[p] == s2[p - t] for p in range(max(0, t), min(m, t + n))):
                continue                                   # có vị trí cùng phủ mà khác ký tự
            lo, hi = min(0, t), max(m, t + n)
            merged = "".join(s1[p] if 0 <= p < m else s2[p - t] for p in range(lo, hi))
            if best is None or len(merged) < len(best):
                best = merged
        return best

print(SolutionOffset().shortestSuperstring("aba", "bab"))
print(SolutionOffset().shortestSuperstring("aa", "aaa"))
print(SolutionOffset().shortestSuperstring("ab", "ca"))

baba
aaa
cab


Độ phức tạp Cách 3: `m + n` độ lệch × `O(min(m, n))` kiểm tra khớp × `O(m + n)` dựng chuỗi khi khớp -> `O((m + n)²)` thời gian trong trường hợp xấu, `O(m + n)` bộ nhớ. Cách này **không cần** xử lý riêng "nằm trong nhau", nên rất khó sót trường hợp.

### Cách 4 — KMP để tính overlap trong `O(m + n)`

`overlap(a, b) = prefix_function(b + "#" + a)[-1]`. Kết hợp với kiểm tra `in` cho trường hợp chứa nhau. Tổng thời gian tuyến tính.

In [11]:
class SolutionKMP:
    def shortestSuperstring(self, s1, s2):
        if s1 in s2:
            return s2
        if s2 in s1:
            return s1

        def pf(t):
            pi = [0] * len(t)
            for i in range(1, len(t)):
                k = pi[i - 1]
                while k > 0 and t[i] != t[k]:
                    k = pi[k - 1]
                if t[i] == t[k]:
                    k += 1
                pi[i] = k
            return pi

        def merge(a, b):
            k = pf(b + "#" + a)[-1]                       # overlap(a, b)
            return a + b[k:]

        r1, r2 = merge(s1, s2), merge(s2, s1)
        return r1 if len(r1) <= len(r2) else r2

print(SolutionKMP().shortestSuperstring("aba", "bab"))
print(SolutionKMP().shortestSuperstring("aa", "aaa"))
print(SolutionKMP().shortestSuperstring("ab", "ca"))

abab
aaa
cab


Độ phức tạp Cách 4: `pf` chạy `O(m + n)`, hai lần merge -> `O(m + n)` thời gian (cộng chi phí `in`), `O(m + n)` bộ nhớ. Hữu ích khi độ dài lớn hơn nhiều (ví dụ `10^5`).

## 8. TRACE từng bước

Trace Cách 1 cho `s1 = "aba"`, `s2 = "bab"`. Không chứa nhau (`"aba"` không nằm trong `"bab"` và ngược lại). Thứ tự `(s1, s2)`:

| `k` | hậu tố `a[len(a)-k:]` | tiền tố `b[:k]` | bằng nhau? |
|---|---|---|---|
| 3 | `aba` | `bab` | không |
| 2 | `ba` | `ba` | **có** -> overlap `2` |

Gộp `a + b[2:] = "aba" + "b" = "abab"` (dài `4`). Thứ tự `(s2, s1)`: `k = 3`: `bab` vs `aba` không; `k = 2`: `ab` vs `ab` có -> `"bab" + "a" = "baba"` (dài `4`). Hòa độ dài -> lấy kết quả đầu: `"abab"`.

In [12]:
def trace_merge(a, b):
    print(f"ghép {a!r} rồi {b!r}:")
    for k in range(min(len(a), len(b)), -1, -1):
        suffix, prefix = a[len(a) - k:], b[:k]
        print(f"   k={k}: hậu tố {suffix!r} vs tiền tố {prefix!r} -> {'KHỚP' if suffix == prefix else 'không'}")
        if suffix == prefix:
            res = a + b[k:]
            print(f"   => overlap {k}, kết quả {res!r} (dài {len(res)})")
            return res

for s1, s2 in [("aba", "bab"), ("ab", "ca")]:
    r1, r2 = trace_merge(s1, s2), trace_merge(s2, s1)
    print("chọn:", min(r1, r2, key=len), "\n")

ghép 'aba' rồi 'bab':
   k=3: hậu tố 'aba' vs tiền tố 'bab' -> không
   k=2: hậu tố 'ba' vs tiền tố 'ba' -> KHỚP
   => overlap 2, kết quả 'abab' (dài 4)
ghép 'bab' rồi 'aba':
   k=3: hậu tố 'bab' vs tiền tố 'aba' -> không
   k=2: hậu tố 'ab' vs tiền tố 'ab' -> KHỚP
   => overlap 2, kết quả 'baba' (dài 4)
chọn: abab 

ghép 'ab' rồi 'ca':
   k=2: hậu tố 'ab' vs tiền tố 'ca' -> không
   k=1: hậu tố 'b' vs tiền tố 'c' -> không
   k=0: hậu tố '' vs tiền tố '' -> KHỚP
   => overlap 0, kết quả 'abca' (dài 4)
ghép 'ca' rồi 'ab':
   k=2: hậu tố 'ca' vs tiền tố 'ab' -> không
   k=1: hậu tố 'a' vs tiền tố 'a' -> KHỚP
   => overlap 1, kết quả 'cab' (dài 3)
chọn: cab 



## 9. Code nộp LeetCode

In [13]:
class Solution:
    def shortestSuperstring(self, s1: str, s2: str) -> str:
        if s1 in s2:
            return s2
        if s2 in s1:
            return s1

        def merge(a: str, b: str) -> str:
            for k in range(min(len(a), len(b)), -1, -1):
                if a[len(a) - k:] == b[:k]:
                    return a + b[k:]
            return a + b

        return min(merge(s1, s2), merge(s2, s1), key=len)

print(Solution().shortestSuperstring("aba", "bab"), Solution().shortestSuperstring("aa", "aaa"))

abab aaa


## 10. Vì sao thuật toán đúng

- **Mọi đáp án là một cách đặt hai chuỗi lên đường thẳng:** chuỗi chứa cả `s1` và `s2` là chuỗi có một vị trí bắt đầu của `s1` và một vị trí bắt đầu của `s2`. Bỏ các ký tự thừa không thuộc `s1` hay `s2` ta được chuỗi ngắn hơn hoặc bằng, nên đáp án ngắn nhất được xác định bởi **độ lệch tương đối `t`**, và độ dài là `max(m, t+n) - min(0, t)`. (Cách 3 vét cạn đúng tập này.)
- **Tồn tại nằm trọn:** nếu một chuỗi nằm trọn trong chuỗi kia thì đáp án là chuỗi dài hơn (độ dài `max(m, n)`, nhỏ nhất có thể vì phải chứa chuỗi dài hơn).
- **Ngược lại (không chứa nhau):** `s2` thò ra bên phải hoặc bên trái `s1` (hoặc nối thẳng). Với `s2` thò ra phải, phần cùng phủ là hậu tố của `s1` và tiền tố của `s2` có độ dài `k` (khớp), độ dài tổng `m + n - k`; **`k` càng lớn càng ngắn**, nên ta chọn `k = overlap(s1, s2)` lớn nhất. Bên trái là đối xứng với `overlap(s2, s1)`. Lấy min của hai là đáp án.
- **Overlap bằng 0** luôn là một giá trị hợp lệ (hậu tố/tiền tố rỗng), đảm bảo vòng lặp `k` luôn trả kết quả (nối thẳng).
- **KMP:** `pi[-1]` của `b + "#" + a` là tiền tố dài nhất của `b#a` cũng là hậu tố của nó; do `#` không xuất hiện trong chữ thường nên tiền tố đó nằm gọn trong `b`, tức đúng overlap(a, b).
- **Bộ test:** so sánh với một **oracle vét cạn** (thử mọi chuỗi trên alphabet nhỏ theo độ dài tăng dần) trên các cặp chuỗi nhỏ, và đối chiếu bốn cách trên chuỗi lớn hơn.

## 11. Phân tích Time / Space Complexity

Gọi `m = len(s1)`, `n = len(s2)`.

| Cách | Time | Space |
|---|---|---|
| 1. Overlap từ lớn xuống nhỏ | `O(min(m, n)²)` (+ `in`) | `O(m + n)` |
| 2. Quét hậu tố bằng `startswith` | `O(m · min(m, n))` (+ `in`) | `O(m + n)` |
| 3. Trượt theo mọi độ lệch | `O((m + n)²)` | `O(m + n)` |
| 4. KMP | `O(m + n)` | `O(m + n)` |

Với `m, n <= 100` mọi cách tức thì. KMP là cách mở rộng được khi chuỗi dài; Cách 1 là lựa chọn dễ viết, ít sai khi thi.

## 12. Bộ test

Vì có **nhiều đáp án đúng**, ta kiểm tra theo hai tiêu chí: (a) kết quả **chứa cả `s1` và `s2`**; (b) **độ dài** bằng độ dài tối ưu. Độ dài tối ưu tính bằng công thức `max(m, n)` nếu chứa nhau, ngược lại `m + n - max(overlap(s1, s2), overlap(s2, s1))`; công thức này được kiểm chứng bằng **oracle vét cạn** ở dưới.

Test 1: ví dụ đề và các edge case (hai chuỗi bằng nhau, một chuỗi 1 ký tự, không chồng lấp, chứa nhau ở giữa/đầu/cuối, chồng lấp cả hai phía).

In [14]:
from itertools import product

def valid(s1, s2, ans):
    return s1 in ans and s2 in ans

def formula_len(s1, s2):
    if s1 in s2 or s2 in s1:
        return max(len(s1), len(s2))
    return len(s1) + len(s2) - max(overlap(s1, s2), overlap(s2, s1))

def brute_len(s1, s2):
    # oracle vét cạn: thử mọi chuỗi trên alphabet của hai chuỗi, độ dài tăng dần
    alphabet = sorted(set(s1 + s2))
    L = max(len(s1), len(s2))
    while True:
        for tup in product(alphabet, repeat=L):
            cand = "".join(tup)
            if s1 in cand and s2 in cand:
                return L
        L += 1

impls = [SolutionOverlap(), SolutionSuffixScan(), SolutionOffset(), SolutionKMP(), Solution()]
tests = [
    ("aba", "bab", 4),
    ("aa", "aaa", 3),
    ("a", "a", 1),                    # hai chuỗi bằng nhau
    ("a", "b", 2),                    # không chồng lấp
    ("abc", "b", 3),                  # s2 nằm giữa s1
    ("abc", "ab", 3),                 # s2 là tiền tố của s1
    ("abc", "bc", 3),                 # s2 là hậu tố của s1
    ("ab", "ca", 3),                  # chỉ ghép s2 trước s1 mới ngắn: "cab"
    ("baaa", "aaac", 5),              # overlap lớn nhất 3 -> "baaac"
    ("abcd", "cdef", 6),              # "abcdef"
    ("xyz", "abc", 6),                # không chồng lấp
    ("ab", "ba", 3),                  # "aba" hoặc "bab"
]
for s1, s2, exp in tests:
    assert 1 <= len(s1) <= 100 and 1 <= len(s2) <= 100 and s1.islower() and s2.islower()
    assert formula_len(s1, s2) == exp == brute_len(s1, s2), (s1, s2, formula_len(s1, s2), exp)
    for im in impls:
        ans = im.shortestSuperstring(s1, s2)
        assert valid(s1, s2, ans), (type(im).__name__, s1, s2, ans)
        assert len(ans) == exp, (type(im).__name__, s1, s2, ans, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 12


Test 2: **vét cạn so với oracle** — mọi cặp `(s1, s2)` trên alphabet `{a, b}` với độ dài `1..3` mỗi chuỗi (tổng `14 × 14 = 196` cặp): độ dài của cả năm cài đặt phải bằng độ dài `brute_len`.

In [15]:
small = ["".join(t) for L in range(1, 4) for t in product("ab", repeat=L)]
count = 0
for s1 in small:
    for s2 in small:
        opt = brute_len(s1, s2)
        assert formula_len(s1, s2) == opt, (s1, s2)
        for im in impls:
            ans = im.shortestSuperstring(s1, s2)
            assert valid(s1, s2, ans) and len(ans) == opt, (type(im).__name__, s1, s2, ans, opt)
        count += 1
print("vét cạn", count, "cặp chuỗi nhỏ OK")

vét cạn 196 cặp chuỗi nhỏ OK


Test 3: ngẫu nhiên có seed, độ dài đến `100`, alphabet nhỏ (`ab`, `abc`) để overlap thường xuyên xuất hiện, kèm các ca **cài sẵn** chồng lấp (cắt hậu tố của `s1` làm tiền tố của `s2`). Kiểm tra: chứa cả hai, độ dài bằng `formula_len`, và bốn cách cho cùng độ dài.

In [16]:
import random
random.seed(3571)
for t in range(1500):
    alpha = random.choice(["ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    s1 = "".join(random.choice(alpha) for _ in range(random.randint(1, 100)))
    s2 = "".join(random.choice(alpha) for _ in range(random.randint(1, 100)))
    if t % 3 == 0:                                         # cài sẵn chồng lấp
        k = random.randint(1, min(len(s1), 50))
        extra = "".join(random.choice(alpha) for _ in range(random.randint(0, 50)))
        s2 = s1[len(s1) - k:] + extra
        s2 = s2[:100] if s2 else s1[-1]
    assert 1 <= len(s1) <= 100 and 1 <= len(s2) <= 100
    opt = formula_len(s1, s2)
    for im in impls:
        ans = im.shortestSuperstring(s1, s2)
        assert valid(s1, s2, ans) and len(ans) == opt, (type(im).__name__, s1, s2)
print("1500 test ngẫu nhiên OK")

1500 test ngẫu nhiên OK


Test 4: tính chất — kết quả luôn thỏa `max(m, n) <= len <= m + n`; hoán đổi `s1`, `s2` không đổi độ dài; một chuỗi ghép với chính nó cho đúng chính nó.

In [17]:
random.seed(71)
for _ in range(1000):
    s1 = "".join(random.choice("abc") for _ in range(random.randint(1, 100)))
    s2 = "".join(random.choice("abc") for _ in range(random.randint(1, 100)))
    a, b = Solution().shortestSuperstring(s1, s2), Solution().shortestSuperstring(s2, s1)
    assert max(len(s1), len(s2)) <= len(a) <= len(s1) + len(s2)
    assert len(a) == len(b)
    assert Solution().shortestSuperstring(s1, s1) == s1
print("tính chất (cận độ dài, đối xứng, tự ghép): OK")

tính chất (cận độ dài, đối xứng, tự ghép): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ nối `s1 + s2`

In [18]:
def wrong_concat(s1, s2):
    return s1 + s2                                   # SAI: không tận dụng chồng lấp

s1, s2 = "aba", "bab"
print("sai :", wrong_concat(s1, s2), len(wrong_concat(s1, s2)))
print("đúng:", Solution().shortestSuperstring(s1, s2), len(Solution().shortestSuperstring(s1, s2)))
assert len(wrong_concat(s1, s2)) != len(Solution().shortestSuperstring(s1, s2))

sai : ababab 6
đúng: abab 4


Vì sao sai: `"ababab"` hợp lệ nhưng dài `6`; chồng lấp `"ba"` cho `"abab"` dài `4`. **Sửa:** tìm overlap lớn nhất và gộp.

### Lỗi 2: bỏ qua trường hợp một chuỗi nằm trong chuỗi kia

In [19]:
def wrong_no_containment(s1, s2):
    def merge(a, b):
        for k in range(min(len(a), len(b)), -1, -1):
            if a[len(a) - k:] == b[:k]:
                return a + b[k:]
    return min(merge(s1, s2), merge(s2, s1), key=len)    # SAI: thiếu kiểm tra `in` trước

s1, s2 = "abc", "b"
print("sai :", wrong_no_containment(s1, s2))
print("đúng:", Solution().shortestSuperstring(s1, s2))
assert len(wrong_no_containment(s1, s2)) != len(Solution().shortestSuperstring(s1, s2))

sai : abcb
đúng: abc


Vì sao sai: `"b"` nằm **giữa** `"abc"` (không phải hậu tố/tiền tố), nên phép dán hậu tố–tiền tố không phát hiện ra và ra `"abcb"` (dài 4) thay vì `"abc"` (dài 3). **Sửa:** kiểm tra `s1 in s2` và `s2 in s1` trước.

### Lỗi 3: chỉ thử một thứ tự ghép

In [20]:
def wrong_one_order(s1, s2):
    if s1 in s2:
        return s2
    if s2 in s1:
        return s1
    for k in range(min(len(s1), len(s2)), -1, -1):
        if s1[len(s1) - k:] == s2[:k]:
            return s1 + s2[k:]                           # SAI: không thử s2 rồi s1

s1, s2 = "ab", "ca"
print("sai :", wrong_one_order(s1, s2))
print("đúng:", Solution().shortestSuperstring(s1, s2))
assert len(wrong_one_order(s1, s2)) != len(Solution().shortestSuperstring(s1, s2))

sai : abca
đúng: cab


Vì sao sai: `overlap("ab", "ca") = 0` nên ra `"abca"` (dài 4), trong khi đặt `"ca"` trước `"ab"` có `overlap("ca", "ab") = 1` cho `"cab"` (dài 3). **Sửa:** thử cả `(s1, s2)` và `(s2, s1)`, lấy ngắn hơn.

### Lỗi 4: chọn overlap **nhỏ nhất** (khác 0) thay vì lớn nhất

In [21]:
def wrong_smallest_overlap(s1, s2):
    if s1 in s2:
        return s2
    if s2 in s1:
        return s1
    def merge(a, b):
        for k in range(1, min(len(a), len(b)) + 1):       # SAI: duyệt k tăng dần -> gặp overlap nhỏ trước
            if a[len(a) - k:] == b[:k]:
                return a + b[k:]
        return a + b
    return min(merge(s1, s2), merge(s2, s1), key=len)

s1, s2 = "baaa", "aaac"
print("sai :", wrong_smallest_overlap(s1, s2), len(wrong_smallest_overlap(s1, s2)))
print("đúng:", Solution().shortestSuperstring(s1, s2), len(Solution().shortestSuperstring(s1, s2)))
assert len(wrong_smallest_overlap(s1, s2)) != len(Solution().shortestSuperstring(s1, s2))

sai : baaaaac 7
đúng: baaac 5


Vì sao sai: duyệt `k` từ `1` lên thì gặp `k = 1` (`"a" == "a"`) trước, ra `"baaaaac"` (dài 7); overlap lớn nhất là `3` cho `"baaac"` (dài 5). **Sửa:** duyệt `k` **từ lớn xuống nhỏ** và dừng ở lần khớp đầu tiên.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi **ngắn nhất chứa** các chuỗi cho trước" -> **overlap** hậu tố–tiền tố; với hai chuỗi chỉ cần thử hai thứ tự.
- Luôn kiểm tra **chứa nhau** (`in`) trước.
- Overlap lớn nhất = duyệt `k` **giảm dần** và dừng ngay khi khớp.
- Cẩn thận `s[-0:]`: dùng `s[len(s) - k:]` để `k = 0` cho chuỗi rỗng.
- Khi có **nhiều đáp án đúng**, test bằng **tiêu chí** (chứa cả hai + độ dài) thay vì so chuỗi.
- Bài họ hàng: `943. Find the Shortest Superstring` (Hard, nhiều chuỗi: quy hoạch động bitmask trên đồ thị overlap), `28. Find the Index of the First Occurrence in a String`, `459. Repeated Substring Pattern`.

## 15. Biến thể của bài

1. **Bản nhiều chuỗi (`943. Find the Shortest Superstring`, Hard):** cho một **mảng** chuỗi, tìm chuỗi ngắn nhất chứa tất cả — duyệt thứ tự bằng bitmask DP với trọng số overlap.
2. Trả về **độ dài** ngắn nhất thay vì chuỗi (chỉ cần `m + n - max overlap`).
3. Trả về chuỗi **nhỏ nhất theo thứ tự từ điển** trong số các đáp án ngắn nhất.
4. Chuỗi **con không liên tiếp** (superstring theo subsequence): đó là bài *Shortest Common Supersequence* (`1092`, quy hoạch động LCS).
5. `s1`, `s2` rất dài (`10^6`): dùng KMP hoặc Z-function cho cả phép `in` và overlap.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Overlap từ lớn xuống nhỏ | thử `k` giảm dần, dán | `O(min(m,n)²)` | `O(m+n)` | **dễ viết, nên nộp** |
| 2. Quét hậu tố `startswith` | điểm bắt đầu hậu tố tăng dần | `O(m · min(m,n))` | `O(m+n)` | thay đổi góc nhìn, ít cắt lát |
| 3. Trượt theo mọi độ lệch | vét cạn vị trí tương đối | `O((m+n)²)` | `O(m+n)` | không sót trường hợp chứa nhau |
| 4. KMP | overlap = `pi[-1]` của `b#a` | `O(m+n)` | `O(m+n)` | chuỗi rất dài |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `len(shortestSuperstring("ab", "ba"))` bằng bao nhiêu?

**Bài 2 (tính tay):** `s1 = "abcd"`, `s2 = "cdef"`: overlap hai chiều và chuỗi ngắn nhất.

**Bài 3 (điền TODO):** hoàn thành `overlap_ex(a, b)` trả về độ chồng lấp lớn nhất giữa hậu tố của `a` và tiền tố của `b`.

**Bài 4 (tự viết lại):** viết `shortest_len(s1, s2)` trả về **chỉ độ dài** của chuỗi ngắn nhất (không dựng chuỗi), dùng công thức.

**Bài 5 (mini challenge):** viết `shortest_superstring_three(a, b, c)` tìm chuỗi ngắn nhất chứa **ba** chuỗi bằng cách thử mọi thứ tự (`permutations`) và dán bằng overlap (giả sử không chuỗi nào chứa chuỗi khác).

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(len(Solution().shortestSuperstring("ab", "ba")))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def overlap_ex(a, b):
    # TODO: thử k từ min(len(a), len(b)) giảm dần, trả k đầu tiên có a[len(a)-k:] == b[:k]
    return 0

3


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** độ dài = `m + n - overlap`; overlap là phần hậu tố của chuỗi trước trùng tiền tố của chuỗi sau.

**Mức 2 (vừa):** Bài 3: `for k in range(min(len(a), len(b)), -1, -1): if a[len(a)-k:] == b[:k]: return k`. Bài 4: nếu chứa nhau thì `max(m, n)`, ngược lại `m + n - max(overlap(s1,s2), overlap(s2,s1))`. Bài 5: với mỗi hoán vị `(x, y, z)`, gộp `x` với `y` rồi gộp kết quả với `z`; lấy ngắn nhất.

**Mức 3 (gần đáp án):** Bài 1: `"ab"` và `"ba"`: overlap `1` ở cả hai chiều -> `2 + 2 - 1 = 3` (ví dụ `"aba"`). Bài 2: hậu tố `"cd"` của `"abcd"` trùng tiền tố `"cd"` của `"cdef"` -> overlap `2`; chiều ngược lại overlap `0`; kết quả `"abcdef"` dài `6`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
from itertools import permutations

# Bài 1
assert len(Solution().shortestSuperstring("ab", "ba")) == 3

# Bài 2
assert overlap("abcd", "cdef") == 2 and overlap("cdef", "abcd") == 0
assert Solution().shortestSuperstring("abcd", "cdef") == "abcdef"

# Bài 3
def overlap_ex(a, b):
    for k in range(min(len(a), len(b)), -1, -1):
        if a[len(a) - k:] == b[:k]:
            return k
assert overlap_ex("aba", "bab") == 2 and overlap_ex("ab", "ca") == 0 and overlap_ex("ca", "ab") == 1

# Bài 4
def shortest_len(s1, s2):
    if s1 in s2 or s2 in s1:
        return max(len(s1), len(s2))
    return len(s1) + len(s2) - max(overlap_ex(s1, s2), overlap_ex(s2, s1))
assert shortest_len("aba", "bab") == 4 and shortest_len("aa", "aaa") == 3 and shortest_len("ab", "ca") == 3

# Bài 5
def shortest_superstring_three(a, b, c):
    def merge(x, y):
        k = overlap_ex(x, y)
        return x + y[k:]
    best = None
    for x, y, z in permutations([a, b, c]):
        cand = merge(merge(x, y), z)
        if best is None or len(cand) < len(best):
            best = cand
    return best
res = shortest_superstring_three("ab", "bc", "cd")
assert res == "abcd" and all(w in res for w in ("ab", "bc", "cd"))
res2 = shortest_superstring_three("xy", "yz", "zw")
assert len(res2) == 4 and all(w in res2 for w in ("xy", "yz", "zw"))
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "chuỗi ngắn nhất chứa hai chuỗi":**

```python
if s1 in s2: return s2
if s2 in s1: return s1

def merge(a, b):
    for k in range(min(len(a), len(b)), -1, -1):      # overlap lớn nhất trước
        if a[len(a) - k:] == b[:k]:
            return a + b[k:]

return min(merge(s1, s2), merge(s2, s1), key=len)
```

**Checklist trước khi submit:**

- [ ] Đã kiểm tra một chuỗi **nằm trong** chuỗi kia (kể cả nằm giữa) trước?
- [ ] Thử **cả hai thứ tự** `(s1, s2)` và `(s2, s1)`?
- [ ] Duyệt `k` **từ lớn xuống nhỏ** và dừng ở lần khớp đầu (overlap lớn nhất)?
- [ ] Dùng `s[len(s) - k:]` (không dùng `s[-k:]` khi `k` có thể bằng `0`)?
- [ ] Đã thử `("aba","bab")`, `("aa","aaa")`, `("ab","ca")`, `("abc","b")`?
- [ ] Test theo tiêu chí (chứa cả hai + độ dài), vì có nhiều đáp án đúng?
- [ ] Chữ ký đúng `shortestSuperstring(self, s1: str, s2: str) -> str`?